# Test decode_N.py: Data Generation and Execution

This notebook generates realistic test data and calls the `decode_N.py` script to demonstrate end-to-end decoding workflow.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tempfile
import os
import sys

# Add parent directory to path to import Fast_functions
sys.path.insert(0, '/Users/ltalamanca/My Drive/Git/PoolPy')

from Fast_functions import *
print("✓ All required libraries imported successfully")

✓ All required libraries imported successfully


## Step 1: Generate Test Data

Generate realistic continuous readouts and well assignment matrix, then save as CSV files.


In [2]:
import subprocess
import os
import importlib
import sys

# Reload Fast_functions to ensure sparse_uniform_array is available
if 'Fast_functions' in sys.modules:
    importlib.reload(sys.modules['Fast_functions'])
    
# Explicitly import the function
from Fast_functions import sparse_uniform_array, assign_wells_bin

print("=" * 70)
print("GENERATING TEST DATA FOR decode_N.py")
print("=" * 70)

# Parameters
n_compounds = 20
n_active = max(1, n_compounds // 4)  # 25% of compounds
output_dir = "/Users/ltalamanca/My Drive/Git/PoolPy"

print(f"\nParameters:")
print(f"  Number of compounds: {n_compounds}")
print(f"  Active compounds per readout: {n_active}")
print(f"  Output directory: {output_dir}")

# 1. Generate and save WA matrix
print(f"\n1. Generating WA matrix...")
WA_int = assign_wells_bin(n_compounds=n_compounds, differentiate=2)
wa_df = pd.DataFrame(WA_int, 
                     columns=[f"Pool_{i}" for i in range(WA_int.shape[1])], 
                     index=[f"Compound_{i}" for i in range(WA_int.shape[0])])
wa_path = os.path.join(output_dir, "test_wa_notebook.csv")
wa_df.to_csv(wa_path)
print(f"   ✓ Saved WA matrix to {os.path.basename(wa_path)}")
print(f"   Shape: {WA_int.shape} ({WA_int.shape[0]} compounds, {WA_int.shape[1]} pools)")

# 2. Generate and save continuous readouts (with header and index)
print(f"\n2. Generating continuous readouts...")
continuous_readouts = []
readout_ids = []
for i in range(3):
    # Generate ground truth (25% of compounds active)
    gt = sparse_uniform_array(n_compounds, n_active, low=0.5, high=2.5, random_state=42+i)
    
    # Multiply by WA to get continuous readout
    readout = gt.dot(WA_int)
    
    # Add Gaussian noise
    noise = np.random.normal(0, 0.1, readout.shape)
    readout_noisy = readout + noise
    
    continuous_readouts.append(readout_noisy)
    readout_ids.append(f"Sample_{i+1}")
    print(f"   Sample {i+1}: {np.round(readout_noisy, 3)}")

# Save to CSV with header and index
readouts_df = pd.DataFrame(continuous_readouts, 
                           columns=[f"Pool_{i}" for i in range(WA_int.shape[1])],
                           index=readout_ids)
readouts_path = os.path.join(output_dir, "test_continuous_notebook.csv")
readouts_df.to_csv(readouts_path)
print(f"   ✓ Saved continuous readouts to {os.path.basename(readouts_path)}")
print(f"   Format: CSV with header and sample IDs as index")

# 3. Generate and save binary readouts (pool indices) - BOTH UNQUOTED AND QUOTED FORMATS
print(f"\n3. Generating binary readouts...")
binary_readouts = []
for i in range(3):
    # Random positive compounds
    n_pos = np.random.randint(1, 4)
    pos_compounds = np.random.choice(n_compounds, n_pos, replace=False).tolist()
    
    # Get positive pools from WA
    pos_pools = []
    for pool_idx in range(WA_int.shape[1]):
        if any(WA_int[comp, pool_idx] for comp in pos_compounds):
            pos_pools.append(pool_idx)
    
    binary_readouts.append(','.join(map(str, pos_pools)))
    print(f"   Sample {i+1}: Positive pools {pos_pools}")

# 3a. Save UNQUOTED binary format (Sample_1,1,2,3)
print(f"\n3a. Saving UNQUOTED binary format...")
binary_path_unquoted = os.path.join(output_dir, "test_binary_notebook_unquoted.csv")
with open(binary_path_unquoted, 'w') as f:
    for i, pools_str in enumerate(binary_readouts):
        f.write(f"Sample_{i+1},{pools_str}\n")
print(f"   ✓ Saved to {os.path.basename(binary_path_unquoted)}")
print(f"   Format: Sample_ID,pool_indices (unquoted, comma-separated)")

# 3b. Save QUOTED binary format (Sample_1,"1,2,3")
print(f"\n3b. Saving QUOTED binary format...")
binary_path_quoted = os.path.join(output_dir, "test_binary_notebook_quoted.csv")
with open(binary_path_quoted, 'w') as f:
    for i, pools_str in enumerate(binary_readouts):
        f.write(f'Sample_{i+1},"{pools_str}"\n')
print(f"   ✓ Saved to {os.path.basename(binary_path_quoted)}")
print(f"   Format: Sample_ID,\"pool_indices\" (quoted)")

# 3c. Save PIPE-DELIMITED binary format (Sample_1|1,2,3) - original format
print(f"\n3c. Saving PIPE-DELIMITED binary format...")
binary_path = os.path.join(output_dir, "test_binary_notebook.csv")
with open(binary_path, 'w') as f:
    for i, pools_str in enumerate(binary_readouts):
        f.write(f"Sample_{i+1}|{pools_str}\n")
print(f"   ✓ Saved to {os.path.basename(binary_path)}")
print(f"   Format: Sample_ID|pool_indices (pipe-separated)")

print(f"\n{'='*70}")
print("Data generation complete. Ready to test decode_N.py...")
print(f"{'='*70}")


GENERATING TEST DATA FOR decode_N.py

Parameters:
  Number of compounds: 20
  Active compounds per readout: 5
  Output directory: /Users/ltalamanca/My Drive/Git/PoolPy

1. Generating WA matrix...
   ✓ Saved WA matrix to test_wa_notebook.csv
   Shape: (20, 5) (20 compounds, 5 pools)

2. Generating continuous readouts...
   Sample 1: [1.432 6.625 5.854 3.126 2.067]
   Sample 2: [2.13  3.779 3.352 0.009 2.208]
   Sample 3: [-0.041  4.809  5.265  5.558  6.624]
   ✓ Saved continuous readouts to test_continuous_notebook.csv
   Format: CSV with header and sample IDs as index

3. Generating binary readouts...
   Sample 1: Positive pools [1]
   Sample 2: Positive pools [1, 2, 4]
   Sample 3: Positive pools [1, 2, 3, 4]

3a. Saving UNQUOTED binary format...
   ✓ Saved to test_binary_notebook_unquoted.csv
   Format: Sample_ID,pool_indices (unquoted, comma-separated)

3b. Saving QUOTED binary format...
   ✓ Saved to test_binary_notebook_quoted.csv
   Format: Sample_ID,"pool_indices" (quoted)

3c. 

In [3]:
# Step 2: Execute decode_N.py on All Readout CSVs
# Test decode_N.py on continuous, binary (all formats), and mixed formats

# Test both readout types and binary format variants
readout_configs = [
    {
        'name': 'Continuous Readouts',
        'path': readouts_path,
        'description': 'Float values with header and sample IDs'
    },
    {
        'name': 'Binary Readouts (UNQUOTED)',
        'path': binary_path_unquoted,
        'description': 'Unquoted pool indices (Sample_ID,1,2,3)'
    },
    {
        'name': 'Binary Readouts (QUOTED)',
        'path': binary_path_quoted,
        'description': 'Quoted pool indices (Sample_ID,"1,2,3")'
    },
    {
        'name': 'Binary Readouts (PIPE-DELIMITED)',
        'path': binary_path,
        'description': 'Pipe-delimited format (Sample_ID|1,2,3)'
    }
]

python_exec = "/Users/ltalamanca/uv_2/bin/python"
script_path = os.path.join(output_dir, "decode_N.py")
all_results = {}

for config in readout_configs:
    print(f"\n{'='*70}")
    print(f"Testing: {config['name']}")
    print(f"Description: {config['description']}")
    print(f"{'='*70}")
    
    cmd = [
        python_exec,
        script_path,
        "--path_to_WA", wa_path,
        "--readout", config['path'],
        "--differentiate", str(2),
        "--diluting", "True"
    ]
    
    print(f"\nCommand:")
    print(f"  {' '.join(cmd)}")
    print(f"\nExecuting...")
    
    # Run decode_N.py
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=output_dir)
    
    print(f"\nReturn code: {result.returncode}")
    if result.stdout:
        print(f"\nStdout:\n{result.stdout}")
    if result.stderr:
        print(f"\nStderr:\n{result.stderr}")
    
    all_results[config['name']] = result

# Display all results
print(f"\n{'='*70}")
print("DECODING RESULTS SUMMARY")
print(f"{'='*70}\n")

output_csv = os.path.join(output_dir, "decoded_readouts.csv")
if os.path.exists(output_csv):
    decoded_df = pd.read_csv(output_csv)
    print("Decoded readouts from most recent test:")
    print(decoded_df.to_string(index=False))
    print(f"\n✓ Results saved to {os.path.basename(output_csv)}")
else:
    print(f"⚠ Output file not found: {output_csv}")



Testing: Continuous Readouts
Description: Float values with header and sample IDs

Command:
  /Users/ltalamanca/uv_2/bin/python /Users/ltalamanca/My Drive/Git/PoolPy/decode_N.py --path_to_WA /Users/ltalamanca/My Drive/Git/PoolPy/test_wa_notebook.csv --readout /Users/ltalamanca/My Drive/Git/PoolPy/test_continuous_notebook.csv --differentiate 2 --diluting True

Executing...

Return code: 0

Testing: Binary Readouts (UNQUOTED)
Description: Unquoted pool indices (Sample_ID,1,2,3)

Command:
  /Users/ltalamanca/uv_2/bin/python /Users/ltalamanca/My Drive/Git/PoolPy/decode_N.py --path_to_WA /Users/ltalamanca/My Drive/Git/PoolPy/test_wa_notebook.csv --readout /Users/ltalamanca/My Drive/Git/PoolPy/test_binary_notebook_unquoted.csv --differentiate 2 --diluting True

Executing...

Return code: 0

Testing: Binary Readouts (QUOTED)
Description: Quoted pool indices (Sample_ID,"1,2,3")

Command:
  /Users/ltalamanca/uv_2/bin/python /Users/ltalamanca/My Drive/Git/PoolPy/decode_N.py --path_to_WA /Users/

In [4]:
readouts_path

'/Users/ltalamanca/My Drive/Git/PoolPy/test_continuous_notebook.csv'

In [5]:
output_dir

'/Users/ltalamanca/My Drive/Git/PoolPy'

In [6]:
# Step 3: Validate Results for Continuous Decoding
# Verify that the decoding results are valid and all constraints are satisfied.

print("=" * 70)
print("VALIDATION OF CONTINUOUS DECODING RESULTS")
print("=" * 70)

# Check if continuous results exist (from continuous readout test)
# Re-run continuous test to get continuous results
continuous_cmd = [
    "/Users/ltalamanca/uv_2/bin/python",
    os.path.join(output_dir, "decode_N.py"),
    "--path_to_WA", wa_path,
    "--readout", readouts_path,
    "--differentiate", str(n_compounds),
    "--diluting", "True"
]

print("\nRe-running continuous decoding for validation...")
continuous_result = subprocess.run(continuous_cmd, capture_output=True, text=True, cwd=output_dir)

if os.path.exists(output_csv):
    decoded_df = pd.read_csv(output_csv)
    print(f"\nColumns: {decoded_df.columns.tolist()}\n")
    
    # Filter for continuous results (should be from the continuous test)
    for idx, row in decoded_df.iterrows():
        readout_id = row.get('readout_id') or row.get('Readout Id') or f"Readout {idx+1}"
        decoded_type = row.get('Decoded Type') or row.get('decoded_type') or row.get('Decoded type')
        
        # Only validate if it's a continuous result
        if 'continuous' in str(decoded_type).lower():
            print(f"\nReadout {idx + 1} ({readout_id}):")
            print(f"  Decoded Type: {decoded_type}")
            
            # Parse the decoder output
            output_col = row.get('Decoder Output') or row.get('decoder_output')
            output_str = str(output_col).strip()
            try:
                # Remove brackets and extra whitespace, handle newlines
                output_str = output_str.replace('[', '').replace(']', '').replace('\\n', ' ')
                coefficients = [float(x) for x in output_str.split() if x.strip()]
                
                print(f"  Number of coefficients: {len(coefficients)}")
                print(f"  Min coefficient: {min(coefficients):.10f}")
                print(f"  Max coefficient: {max(coefficients):.10f}")
                print(f"  Non-zero elements: {sum(1 for c in coefficients if c > 1e-10)}")
                print(f"  All non-negative: {all(c >= -1e-10 for c in coefficients)} ✓")
                
            except Exception as e:
                print(f"  Error parsing coefficients: {e}")
    
    print(f"\n{'='*70}")
    print("✓ VALIDATION COMPLETE")
    print(f"{'='*70}")

VALIDATION OF CONTINUOUS DECODING RESULTS

Re-running continuous decoding for validation...

Columns: ['readout_id', 'Decoded Type', 'Decoder Output']


Readout 1 (Sample_1):
  Decoded Type: continuous
  Number of coefficients: 20
  Min coefficient: 0.0000000000
  Max coefficient: 2.8098476383
  Non-zero elements: 5
  All non-negative: True ✓

Readout 2 (Sample_2):
  Decoded Type: continuous
  Number of coefficients: 20
  Min coefficient: 0.0000000000
  Max coefficient: 1.4554446272
  Non-zero elements: 7
  All non-negative: True ✓

Readout 3 (Sample_3):
  Decoded Type: continuous
  Number of coefficients: 20
  Min coefficient: 0.0000000000
  Max coefficient: 2.6731927694
  Non-zero elements: 4
  All non-negative: True ✓

✓ VALIDATION COMPLETE


## Step 4: Test Stacked Readouts (Mixed Binary and Continuous)

Stack binary and continuous readouts into a single CSV and test decode_N.py on mixed data formats.


In [9]:
print("=" * 70)
print("TESTING STACKED READOUTS (MIXED BINARY AND CONTINUOUS)")
print("=" * 70)

# Read the continuous CSV
continuous_df_saved = pd.read_csv(readouts_path, index_col=0)
continuous_lines = []
for sample_id, row in continuous_df_saved.iterrows():
    values_str = ','.join([str(v) for v in row.values])
    continuous_lines.append(f"{sample_id},{values_str}")

print(f"\nContinuous readouts: {len(continuous_lines)} samples")
for i, line in enumerate(continuous_lines, 1):
    print(f"  {i}: {line}")

# Test stacking with UNQUOTED binary format
print(f"\n{'='*70}")
print("STACKED TEST 1: Continuous + Binary (UNQUOTED)")
print(f"{'='*70}\n")

with open(binary_path_unquoted, 'r') as f:
    binary_lines_unquoted = [line.strip() for line in f if line.strip()]

print(f"Binary readouts (UNQUOTED): {len(binary_lines_unquoted)} samples")
for i, line in enumerate(binary_lines_unquoted, 1):
    print(f"  {i}: {line}")

stacked_lines_unquoted = binary_lines_unquoted + continuous_lines
stacked_path_unquoted = os.path.join(output_dir, "test_stacked_notebook_unquoted.csv")
with open(stacked_path_unquoted, 'w') as f:
    f.write('\n'.join(stacked_lines_unquoted))

print(f"\n✓ Saved stacked readouts to {os.path.basename(stacked_path_unquoted)}")
print(f"  Total samples: {len(stacked_lines_unquoted)} (3 binary + 3 continuous)")

# Test decode_N.py on stacked data (unquoted)
print(f"\n{'='*70}")
print("Executing decode_N.py on stacked readouts (UNQUOTED)...")
print(f"{'='*70}\n")

stacked_cmd_unquoted = [
    python_exec,
    script_path,
    "--path_to_WA", wa_path,
    "--readout", stacked_path_unquoted,
    "--differentiate", str(1),
    "--diluting", "True"
]

stacked_result_unquoted = subprocess.run(stacked_cmd_unquoted, capture_output=True, text=True, cwd=output_dir)

print(f"Return code: {stacked_result_unquoted.returncode}")
if stacked_result_unquoted.stdout:
    print(f"\nStdout:\n{stacked_result_unquoted.stdout}")
if stacked_result_unquoted.stderr:
    print(f"\nStderr:\n{stacked_result_unquoted.stderr}")

print(f"\n{'='*70}")
print("STACKED DECODING RESULTS (UNQUOTED)")
print(f"{'='*70}\n")

output_csv = os.path.join(output_dir, "decoded_readouts.csv")
if os.path.exists(output_csv):
    decoded_stacked_df = pd.read_csv(output_csv)
    print("Decoded stacked readouts (UNQUOTED):")
    print(decoded_stacked_df.to_string(index=False))
    print(f"\n✓ Results saved to {os.path.basename(output_csv)}")
else:
    print(f"⚠ Output file not found: {output_csv}")

# Test stacking with QUOTED binary format
print(f"\n{'='*70}")
print("STACKED TEST 2: Continuous + Binary (QUOTED)")
print(f"{'='*70}\n")

with open(binary_path_quoted, 'r') as f:
    binary_lines_quoted = [line.strip() for line in f if line.strip()]

print(f"Binary readouts (QUOTED): {len(binary_lines_quoted)} samples")
for i, line in enumerate(binary_lines_quoted, 1):
    print(f"  {i}: {line}")

stacked_lines_quoted = binary_lines_quoted + continuous_lines
stacked_path_quoted = os.path.join(output_dir, "test_stacked_notebook_quoted.csv")
with open(stacked_path_quoted, 'w') as f:
    f.write('\n'.join(stacked_lines_quoted))

print(f"\n✓ Saved stacked readouts to {os.path.basename(stacked_path_quoted)}")
print(f"  Total samples: {len(stacked_lines_quoted)} (3 binary + 3 continuous)")

# Test decode_N.py on stacked data (quoted)
print(f"\n{'='*70}")
print("Executing decode_N.py on stacked readouts (QUOTED)...")
print(f"{'='*70}\n")

stacked_cmd_quoted = [
    python_exec,
    script_path,
    "--path_to_WA", wa_path,
    "--readout", stacked_path_quoted,
    "--differentiate", str(1),
    "--diluting", "True"
]

stacked_result_quoted = subprocess.run(stacked_cmd_quoted, capture_output=True, text=True, cwd=output_dir)

print(f"Return code: {stacked_result_quoted.returncode}")
if stacked_result_quoted.stdout:
    print(f"\nStdout:\n{stacked_result_quoted.stdout}")
if stacked_result_quoted.stderr:
    print(f"\nStderr:\n{stacked_result_quoted.stderr}")

print(f"\n{'='*70}")
print("STACKED DECODING RESULTS (QUOTED)")
print(f"{'='*70}\n")

if os.path.exists(output_csv):
    decoded_stacked_df = pd.read_csv(output_csv)
    print("Decoded stacked readouts (QUOTED):")
    print(decoded_stacked_df.to_string(index=False))
    print(f"\n✓ Results saved to {os.path.basename(output_csv)}")
else:
    print(f"⚠ Output file not found: {output_csv}")


TESTING STACKED READOUTS (MIXED BINARY AND CONTINUOUS)

Continuous readouts: 3 samples
  1: Sample_1,1.4316329573374282,6.624945161178971,5.853981614976265,3.126039397978082,2.066533359232104
  2: Sample_2,2.1301645426766584,3.779467158405717,3.352384879724346,0.0089897147350341,2.2078137005277836
  3: Sample_3,-0.0411158920081959,4.808690026516602,5.265226432520218,5.55783592801597,6.624322276101496

STACKED TEST 1: Continuous + Binary (UNQUOTED)

Binary readouts (UNQUOTED): 3 samples
  1: Sample_1,1
  2: Sample_2,1,2,4
  3: Sample_3,1,2,3,4

✓ Saved stacked readouts to test_stacked_notebook_unquoted.csv
  Total samples: 6 (3 binary + 3 continuous)

Executing decode_N.py on stacked readouts (UNQUOTED)...

Return code: 0

STACKED DECODING RESULTS (UNQUOTED)

Decoded stacked readouts (UNQUOTED):
readout_id Decoded Type                                                                                                                                                                          D